In [1]:
import pandas as pd
import sqlite3

In [2]:
from google.colab import drive
drive.mount('/content/drive')
df_turismo = pd.read_csv('/content/drive/MyDrive/tourism_dataset_5000.csv')

Mounted at /content/drive


In [3]:
# MapReduce: conta quantas vezes cada local aparece na coluna de lugares visitados

# Etapa Mapper: transforma cada local visitado em um par no formato (local, 1)
def mapper(linhas):

    # Lista que armazenará os pares gerados pelo Mapper
    pares = []

    # Percorre cada linha recebida
    for linha in linhas:

        # Verifica se a linha não possui um valor nulo
        if pd.notna(linha):

            # Remove colchetes e aspas e separa os locais pela vírgula
            for site in linha.replace("[", "").replace("]", "").replace("'", "").split(","):

                # Remove espaços em branco antes e depois do nome do local
                site = site.strip()

                # Adiciona o local com o valor 1, indicando uma ocorrência
                pares.append((site, 1))

    # Retorna todos os pares gerados
    return pares


# Etapa Shuffle: agrupa os valores que possuem a mesma chave
def shuffle(mapped_data):

    # Dicionário que armazenará os locais e suas listas de ocorrências
    agrupado = {}

    # Percorre cada par formado pelo Mapper
    for chave, valor in mapped_data:

        # Cria uma lista vazia caso o local ainda não esteja no dicionário
        if chave not in agrupado:
            agrupado[chave] = []

        # Adiciona o valor à lista correspondente ao local
        agrupado[chave].append(valor)

    # Retorna os dados agrupados por local
    return agrupado


# Etapa Reducer: soma as ocorrências de cada local
def reducer(agrupado):

    # Cria um dicionário contendo cada local e seu total de ocorrências
    return {
        chave: sum(valores)
        for chave, valores in agrupado.items()
    }

In [4]:
# Usar a coluna "Sites Visited" do DataFrame real
entrada_sites = df_turismo["Sites Visited"].dropna().tolist()
mapeado = mapper(entrada_sites)
agrupado = shuffle(mapeado)
resultado = reducer(agrupado)
print("\n[MapReduce] Lugares mais visitados:")
print(resultado)


[MapReduce] Lugares mais visitados:
{'Eiffel Tower': 2072, 'Great Wall of China': 1991, 'Taj Mahal': 2012, 'Machu Picchu': 1920, 'Colosseum': 1994}


In [5]:
# YARN: simula o gerenciamento e a execução de jobs simples

# Lista que armazenará o histórico dos jobs executados
jobs = []


# Função responsável por enviar e executar um job
def enviar_job(nome, funcao):

    # Exibe uma mensagem informando o início do job
    print(f"\n[YARN] Iniciando job: {nome}")

    # Executa a função associada ao job
    resultado = funcao()

    # Adiciona as informações do job ao histórico
    jobs.append({
        'nome': nome,
        'status': 'finalizado',
        'resultado': resultado
    })

    # Retorna o resultado produzido pelo job
    return resultado


# Define um job que calcula a média de satisfação por faixa etária
def job_media_satisfacao_por_idade():

    # Seleciona apenas as colunas de idade e satisfação
    # e remove registros que possuem valores nulos
    df = df_turismo[["Age", "Satisfaction"]].dropna()

    # Define os limites das faixas etárias
    bins = [0, 18, 30, 45, 60, 80]

    # Define os nomes que serão atribuídos a cada faixa etária
    labels = ['0-18', '19-30', '31-45', '46-60', '61-80']

    # Classifica cada turista em uma faixa etária
    df["FaixaEtaria"] = pd.cut(
        df["Age"],
        bins=bins,
        labels=labels
    )

    # Agrupa os dados por faixa etária
    # Calcula a média de satisfação de cada grupo
    # Arredonda os valores para duas casas decimais
    # Converte o resultado para um dicionário
    resultado = (
        df.groupby("FaixaEtaria", observed=False)["Satisfaction"]
        .mean()
        .round(2)
        .to_dict()
    )

    # Retorna as médias calculadas
    return resultado


# Envia o job para o gerenciador YARN simulado
resultado = enviar_job(
    "Média de Satisfação por Faixa Etária",
    job_media_satisfacao_por_idade
)


# Exibe o título da seção de resultados
print("\n[Resultado do Job]")

# Percorre cada faixa etária e sua respectiva média
for faixa, media in resultado.items():

    # Exibe a média de satisfação da faixa etária
    print(f"Faixa {faixa}: média de satisfação = {media}")


# Exibe o título da seção de histórico
print("\n[YARN] Histórico de Jobs:")

# Percorre todos os jobs armazenados no histórico
for job in jobs:

    # Exibe o nome e o status de cada job executado
    print(f"{job['nome']} - {job['status']}")


[YARN] Iniciando job: Média de Satisfação por Faixa Etária

[Resultado do Job]
Faixa 0-18: média de satisfação = 3.64
Faixa 19-30: média de satisfação = 3.52
Faixa 31-45: média de satisfação = 3.5
Faixa 46-60: média de satisfação = 3.57
Faixa 61-80: média de satisfação = 3.54

[YARN] Histórico de Jobs:
Média de Satisfação por Faixa Etária - finalizado


In [6]:
# Hive: SQL com Pandas — turistas com alta satisfação
print("\n[Hive] Turistas com satisfação ≥ 4:")
df_hive = df_turismo[["Age", "Satisfaction", "Tourist Rating"]]
print(df_hive.query("Satisfaction >= 4"))


[Hive] Turistas com satisfação ≥ 4:
      Age  Satisfaction  Tourist Rating
4      53             4             3.7
6      61             4             4.1
8      34             4             4.0
9      61             4             3.8
13     61             4             4.3
...   ...           ...             ...
4986   68             4             3.7
4991   51             5             5.0
4994   20             4             4.3
4995   31             4             3.9
4996   28             4             4.3

[1939 rows x 3 columns]


In [7]:
# Pig: simula um fluxo de transformação para calcular um índice de experiência
# O índice é obtido multiplicando a idade pelo rating do turista

# Seleciona as colunas de idade e avaliação do turista
# Remove valores nulos, limita aos 100 primeiros registros
# e converte os dados para uma lista
dados_pig = (
    df_turismo[["Age", "Tourist Rating"]]
    .dropna()
    .head(100)
    .values
    .tolist()
)

# Filtra apenas os turistas com rating maior ou igual a 4.0
filtrado = [
    registro
    for registro in dados_pig
    if registro[1] >= 4.0
]

# Transforma cada registro em uma tupla contendo:
# idade, rating e índice de experiência
transformado = [
    (
        int(registro[0]),
        float(registro[1]),
        round(registro[0] * registro[1], 2)
    )
    for registro in filtrado
]

# Exibe o título do resultado
print("\n[Pig] Índice de experiência (idade × rating ≥ 4.0):")

# Exibe o cabeçalho da tabela
print("Idade | Rating | Índice")

# Percorre os registros transformados
for idade, rating, indice in transformado:

    # Exibe os valores formatados em colunas
    print(f"{idade:5d} | {rating:.1f}   | {indice:.2f}")


[Pig] Índice de experiência (idade × rating ≥ 4.0):
Idade | Rating | Índice
   61 | 4.1   | 250.10
   34 | 4.0   | 136.00
   61 | 4.3   | 262.30
   65 | 4.2   | 273.00
   36 | 4.1   | 147.60
   49 | 4.4   | 215.60
   61 | 4.8   | 292.80
   44 | 4.9   | 215.60
   60 | 4.5   | 270.00
   63 | 4.5   | 283.50
   31 | 4.7   | 145.70
   41 | 4.2   | 172.20
   22 | 4.3   | 94.60
   18 | 4.6   | 82.80
   35 | 4.3   | 150.50
   54 | 4.3   | 232.20
   37 | 4.4   | 162.80
   26 | 5.0   | 130.00
   55 | 4.4   | 242.00
   43 | 4.1   | 176.30
   28 | 4.8   | 134.40
   25 | 4.0   | 100.00
   51 | 4.2   | 214.20
   20 | 4.1   | 82.00
   61 | 4.8   | 292.80
   59 | 4.4   | 259.60
   44 | 4.8   | 211.20
   45 | 4.4   | 198.00


In [8]:
# HBase: simula a leitura de um banco NoSQL orientado a colunas

# Cria um dicionário que representa uma tabela do HBase
hbase = {

    # Define a chave de cada registro no formato "turista:id"
    # f-string é usada para inserir valores de variáveis dentro de um texto.
    f"turista:{i}": {

        # Armazena a idade na família de colunas "info"
        "info:idade": int(row["Age"]),

        # Armazena a satisfação do turista
        "info:satisfacao": int(row["Satisfaction"]),

        # Armazena a avaliação atribuída pelo turista
        "info:rating": float(row["Tourist Rating"])
    }

    # Percorre os dez primeiros registros do dataset
    # i representa o índice e row representa os dados da linha
    for i, row in df_turismo.head(10).iterrows()
}

print("\n[HBase] Leitura simulada de dados:")
for chave, colunas in hbase.items():
    print(f"{chave} => idade: {colunas['info:idade']}, satisfação: {colunas['info:satisfacao']}, rating: {colunas['info:rating']}")


[HBase] Leitura simulada de dados:
turista:0 => idade: 48, satisfação: 3, rating: 1.6
turista:1 => idade: 37, satisfação: 3, rating: 2.6
turista:2 => idade: 43, satisfação: 3, rating: 1.7
turista:3 => idade: 46, satisfação: 3, rating: 2.0
turista:4 => idade: 53, satisfação: 4, rating: 3.7
turista:5 => idade: 50, satisfação: 3, rating: 2.2
turista:6 => idade: 61, satisfação: 4, rating: 4.1
turista:7 => idade: 23, satisfação: 3, rating: 2.8
turista:8 => idade: 34, satisfação: 4, rating: 4.0
turista:9 => idade: 61, satisfação: 4, rating: 3.8


In [9]:
# Sqoop: simula a importação e exportação de dados entre um banco relacional e o HDFS

# Cria uma conexão com um banco de dados SQLite armazenado na memória
conn = sqlite3.connect(":memory:")

# Seleciona as colunas desejadas do dataset
# e grava os dados em uma tabela chamada "turistas"
df_turismo[["Tourist ID", "Age", "Satisfaction"]].to_sql(
    "turistas",
    conn,
    index=False
)

# Executa uma consulta SQL para recuperar todos os registros da tabela
df_sql = pd.read_sql(
    "SELECT * FROM turistas",
    conn
)

# Exporta os dados do banco para um arquivo CSV
# O arquivo representa os dados armazenados no HDFS
df_sql.to_csv(
    "turistas_hdfs.csv",
    index=False
)

# Importa novamente os dados do arquivo CSV
# simulando a leitura dos dados armazenados no HDFS
df_hdfs = pd.read_csv("turistas_hdfs.csv")

# Exibe o título da operação realizada
print("\n[Sqoop] Dados exportados/importados para o HDFS:")

# Exibe os dados importados do arquivo
print(df_hdfs)


[Sqoop] Dados exportados/importados para o HDFS:
      Tourist ID  Age  Satisfaction
0              1   48             3
1              2   37             3
2              3   43             3
3              4   46             3
4              5   53             4
...          ...  ...           ...
4995        4996   31             4
4996        4997   28             4
4997        4998   42             3
4998        4999   58             3
4999        5000   32             3

[5000 rows x 3 columns]
